# Perturbation Size Test

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import random
import time
import os

from tqdm import tqdm

from import_shelf import shelf
from shelf.models.resnet.etc import resnet20, resnet32, resnet44, resnet56, resnet110
from shelf.models.transformer import VisionTransformer
from shelf.dataloaders.cifar import get_CIFAR10_dataset
from shelf.trainers.classic import train, validate
from shelf.trainers.zeroth_order import gradient_fo

def cossim_dict(dict_1, dict_2):
    flat_1 = torch.cat([v.view(-1) for v in dict_1.values()])
    flat_2 = torch.cat([v.view(-1) for v in dict_2.values()])
    return F.cosine_similarity(flat_1, flat_2, dim=0)

def dotprod_dict(dict_1, dict_2):
    flat_1 = torch.cat([v.view(-1) for v in dict_1.values()])
    flat_2 = torch.cat([v.view(-1) for v in dict_2.values()])
    return torch.dot(flat_1, flat_2)

## Ablation: Models

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_loader, val_loader = get_CIFAR10_dataset()

x_temp, t_temp = next(iter(train_loader))
x_temp, t_temp = x_temp.to(device), t_temp.to(device)

model_res20 = resnet20().to(device)
model_res32 = resnet32().to(device)
model_res44 = resnet44().to(device)
model_res56 = resnet56().to(device)
model_res110 = resnet110().to(device)

model_tinyvit = VisionTransformer(image_size=32, patch_size=4, num_classes=10, dim=256, depth=4, heads=6, mlp_dim=256).to(device)
model_vit_small = VisionTransformer(image_size=32, patch_size=4, num_classes=10, dim=256, depth=6, heads=8, mlp_dim=512).to(device)
model_vit_base = VisionTransformer(image_size=32, patch_size=4, num_classes=10, dim=512, depth=12, heads=8, mlp_dim=1024).to(device)
model_vit_large = VisionTransformer(image_size=32, patch_size=4, num_classes=10, dim=768, depth=12, heads=12, mlp_dim=3072).to(device)

model_dict = {
    "res20": model_res20,
    "res32": model_res32,
    "res44": model_res44,
    "res56": model_res56,
    "res110": model_res110,
    "tinyvit": model_tinyvit,
    "vit_small": model_vit_small,
    "vit_base": model_vit_base,
    "vit_large": model_vit_large
}

critertion = nn.CrossEntropyLoss()

In [ ]:
def eval_zo_const(input, label, model, criterion, smoothing=1e-3):
    real_gradient_dict = gradient_fo(input, label, model, criterion)

    noise_dict = {pname: torch.randn_like(param) for pname, param in model.named_parameters()}

    real_jvp = dotprod_dict(real_gradient_dict, noise_dict)

    loss_orig = criterion(model(input), label)

    for pname, param in model.named_parameters():
        param.data += smoothing * noise_dict[pname]
    
    loss_perturb = criterion(model(input), label)

    for pname, param in model.named_parameters():
        param.data -= smoothing * noise_dict[pname]

    zo_jvp = (loss_perturb - loss_orig) / smoothing

    return real_jvp, zo_jvp


In [ ]:
print(f"|   MODEL   |  #PARAMS  |   RMEAN   |   ZMEAN   |   RSTD   |   ZSTD   |  CORR  |")

for model_name, model in model_dict.items():
    num_params = sum(p.numel() for p in model.parameters())

    real_jvp_list = []
    zo_jvp_list = []

    # for input, label in tqdm(train_loader, leave=False, desc=f"evaluating {model_name}"):
    for input, label in train_loader:
        input, label = input.to(device), label.to(device)

        real_jvp, zo_jvp = eval_zo_const(input, label, model, critertion, 1e-3)

        real_jvp_list.append(real_jvp.item())
        zo_jvp_list.append(zo_jvp.item())

    real_jvp_list = torch.tensor(real_jvp_list)
    zo_jvp_list = torch.tensor(zo_jvp_list)
    
    r_mean, r_std = real_jvp_list.mean(), real_jvp_list.std()
    z_mean, z_std = zo_jvp_list.mean(), zo_jvp_list.std()

    # correlation between real and zo
    real_jvp_list = (real_jvp_list - real_jvp_list.mean()) / real_jvp_list.std()
    zo_jvp_list = (zo_jvp_list - zo_jvp_list.mean()) / zo_jvp_list.std()

    correlation = torch.dot(real_jvp_list, zo_jvp_list) / len(real_jvp_list)
    
    print(f"| {model_name:9} | {num_params/1e6:7.3f} M | {r_mean:+.2e} | {z_mean:+.2e} | {r_std:.2e} | {z_std:.2e} | {correlation:.4f} |")



## Ablation: Layers

In [9]:
model_res20 = resnet20().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_res20.parameters(), lr=1e-3)
train(train_loader, model_res20, criterion, optimizer, 0)

pnames, params = zip(*model_res20.named_parameters())

for pname, param in model_res20.named_parameters():
    # means of abs of weights
    print(f"{pname} : {param.abs().mean().item():.4e}")

conv1.weight : 2.1533e-01
bn1.weight : 1.0032e+00
bn1.bias : 3.1160e-02
layer1.0.conv1.weight : 9.6527e-02
layer1.0.bn1.weight : 9.9958e-01
layer1.0.bn1.bias : 1.7727e-02
layer1.0.conv2.weight : 9.3965e-02
layer1.0.bn2.weight : 9.9872e-01
layer1.0.bn2.bias : 1.8661e-02
layer1.1.conv1.weight : 9.5179e-02
layer1.1.bn1.weight : 1.0027e+00
layer1.1.bn1.bias : 2.1522e-02
layer1.1.conv2.weight : 9.7957e-02
layer1.1.bn2.weight : 9.9351e-01
layer1.1.bn2.bias : 1.1818e-02
layer1.2.conv1.weight : 9.6322e-02
layer1.2.bn1.weight : 9.9970e-01
layer1.2.bn1.bias : 1.2317e-02
layer1.2.conv2.weight : 9.6203e-02
layer1.2.bn2.weight : 9.8745e-01
layer1.2.bn2.bias : 1.3137e-02
layer2.0.conv1.weight : 9.5622e-02
layer2.0.bn1.weight : 1.0001e+00
layer2.0.bn1.bias : 1.2398e-02
layer2.0.conv2.weight : 6.9087e-02
layer2.0.bn2.weight : 9.8935e-01
layer2.0.bn2.bias : 1.3928e-02
layer2.1.conv1.weight : 6.8029e-02
layer2.1.bn1.weight : 1.0008e+00
layer2.1.bn1.bias : 1.4777e-02
layer2.1.conv2.weight : 6.8894e-02
la